# Skipping the Cache

## What you'll learn

- Force re-execution of cached steps with `skip_cache`
- Apply `skip_cache` per step or pipeline-wide
- Understand when and why to bypass the cache

**Prerequisites:** [Your First Pipeline](../01-getting-started/01-first-pipeline.ipynb),
[Resume and Caching](01-resume-and-caching.ipynb).  
**Estimated time:** 10 minutes  
**GPU required:** No.

---

Artisan's two-level cache automatically skips steps whose inputs
and parameters match a previous successful run. Sometimes you need to force
re-execution, even when the cache key hasn't changed.

| Scenario | Why the cache doesn't help |
|----------|---------------------------|
| Bug fix in operation code | The cache key is based on parameters, not code |
| Non-deterministic operation | You want a fresh sample, not the old one |
| External resource changed | A file or API the operation reads has been updated |
| Fresh pipeline run | Re-execute everything from scratch in the same delta root |

In [ ]:
from __future__ import annotations

from artisan.operations.examples import (
    DataGenerator,
    DataTransformer,
    MetricCalculator,
)
from artisan.orchestration import PipelineManager, Runner, StepDisposition
from artisan.utils import tutorial_setup
from artisan.visualization import inspect_pipeline

In [ ]:
env = tutorial_setup("skip_cache")

## Establish a baseline

Run a three-step pipeline so that later runs have cached results to reuse. Fixed
seeds on both the generator and transformer let us regenerate the same artifact
identities when we bypass the cache.

In [ ]:
pipeline = PipelineManager.create(
    name="skip_cache_demo",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 5, "seed": 42},
    step_runner=Runner.LOCAL,
)
pipeline.run(
    operation=DataTransformer,
    name="transform",
    params={"seed": 100},
    inputs={"dataset": output("generate", "datasets")},
    step_runner=Runner.LOCAL,
)
pipeline.run(
    operation=MetricCalculator,
    name="score",
    inputs={"dataset": output("transform", "dataset")},
    step_runner=Runner.LOCAL,
)

pipeline.finalize()
assert [step.disposition for step in pipeline] == [
    StepDisposition.EXECUTED,
    StepDisposition.EXECUTED,
    StepDisposition.EXECUTED,
]

inspect_pipeline(env.delta_root)

## Confirm caching works

Run the same pipeline again with the same parameters. Every step
should show "CACHED" in the log because nothing changed.

In [ ]:
env = tutorial_setup("skip_cache", clean=False)

pipeline = PipelineManager.create(
    name="cached_run",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 5, "seed": 42},
    step_runner=Runner.LOCAL,
)
pipeline.run(
    operation=DataTransformer,
    name="transform",
    params={"seed": 100},
    inputs={"dataset": output("generate", "datasets")},
    step_runner=Runner.LOCAL,
)
pipeline.run(
    operation=MetricCalculator,
    name="score",
    inputs={"dataset": output("transform", "dataset")},
    step_runner=Runner.LOCAL,
)

pipeline.finalize()
assert [step.disposition for step in pipeline] == [
    StepDisposition.CACHE_HIT,
    StepDisposition.CACHE_HIT,
    StepDisposition.CACHE_HIT,
]

All three results have disposition `CACHE_HIT`: the operations reused their
previous outputs. The assertions check that the tutorial actually exercised
caching, even when notebook output is not saved.

## Skip cache for a single step

Pass `skip_cache=True` to `pipeline.run()` to force re-execution
of that step. Other steps still use the cache normally.

This is the most common pattern — you fixed a bug in one operation
and want to re-run that step without changing its parameters.

In [ ]:
env = tutorial_setup("skip_cache", clean=False)

pipeline = PipelineManager.create(
    name="skip_one_step",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
)
output = pipeline.output

# Step 0: cached (no change)
pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 5, "seed": 42},
    step_runner=Runner.LOCAL,
)

# Step 1: force re-execution
pipeline.run(
    operation=DataTransformer,
    name="transform",
    params={"seed": 100},
    inputs={"dataset": output("generate", "datasets")},
    step_runner=Runner.LOCAL,
    skip_cache=True,
)

# Step 2: cached (no change)
pipeline.run(
    operation=MetricCalculator,
    name="score",
    inputs={"dataset": output("transform", "dataset")},
    step_runner=Runner.LOCAL,
)

pipeline.finalize()
assert [step.disposition for step in pipeline] == [
    StepDisposition.CACHE_HIT,
    StepDisposition.EXECUTED,
    StepDisposition.CACHE_HIT,
]

Steps 0 and 2 have disposition `CACHE_HIT`; step 1 has disposition `EXECUTED`.
The fixed transformer seed recreates the same output identities, so step 2 still
matches its previous inputs. If a rerun changes those identities, downstream
steps must compute with the new inputs.

The executed step’s successful results are persisted and can be reused later.

## Skip cache for the entire pipeline

Pass `skip_cache=True` to `PipelineManager.create()` to bypass the
cache for every step in the pipeline. This is useful after upgrading
a dependency or changing an external resource that affects all steps.

In [ ]:
env = tutorial_setup("skip_cache", clean=False)

pipeline = PipelineManager.create(
    name="fresh_run",
    delta_root=env.delta_root,
    staging_root=env.staging_root,
    working_root=env.working_root,
    skip_cache=True,
)
output = pipeline.output

pipeline.run(
    operation=DataGenerator,
    name="generate",
    params={"count": 5, "seed": 42},
    step_runner=Runner.LOCAL,
)
pipeline.run(
    operation=DataTransformer,
    name="transform",
    params={"seed": 100},
    inputs={"dataset": output("generate", "datasets")},
    step_runner=Runner.LOCAL,
)
pipeline.run(
    operation=MetricCalculator,
    name="score",
    inputs={"dataset": output("transform", "dataset")},
    step_runner=Runner.LOCAL,
)

pipeline.finalize()
assert [step.disposition for step in pipeline] == [
    StepDisposition.EXECUTED,
    StepDisposition.EXECUTED,
    StepDisposition.EXECUTED,
]

No "CACHED" messages — every step re-executed. The pipeline-level
flag overrides any per-step behavior.

## What the bypass changes

`skip_cache=True` bypasses reuse for the selected step. Setting it on the pipeline
applies that choice to every step; a per-step `False` cannot turn caching back on.
Successful results are still persisted. Downstream steps consume the outputs
accepted for this run, including when an artifact ID was already stored.

## Summary

You compared four runs: a baseline, complete cache reuse, a single-step bypass,
and a pipeline-wide bypass. Use a bypass when changed code or external state
makes a cached result unsuitable. Downstream reuse depends on whether the
rerun produces the same input identities for those steps.

## Next steps

- [Resume and Caching](01-resume-and-caching.ipynb) — How caching and resume work together
- [Step Overrides](../05-errors-and-control/01-step-overrides.ipynb) — Other per-step configuration options
- [Execution Flow](../../concepts/execution-flow.md) — How the two-level cache works in the framework